In [164]:
import sys
sys.path.insert(0, '../')

import re
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)

In [38]:
def _str_to_interval(s):
    m = re.match(r'\[([-\d.]+),\s*([-\d.inf]+)\)', str(s))
    if m:
        return pd.Interval(float(m.group(1)), float(m.group(2)), closed='left')
    return s

In [361]:
def _collapse_to_health_bracket(iv):
    """Map a 5-year age bracket to the matching health-survey bracket and the
    fraction of that bracket which falls inside it. Returns (target, weight)
    or None for brackets below 18 that have no health counterpart."""
    if iv == pd.Interval(15.0, 19.0, closed='left'):
        # health starts at 18, so only ages 18-19 of the [15, 20) span count
        return pd.Interval(18.0, 35.0, closed='left'), 2 / 5
    left = iv.left
    if left < 18:
        return None
    if left < 34:
        return pd.Interval(18.0, 35.0, closed='left'), 1.0
    if left < 49:
        return pd.Interval(35.0, 50.0, closed='left'), 1.0
    if left < 64:
        return pd.Interval(50.0, 65.0, closed='left'), 1.0
    return pd.Interval(65.0, np.inf, closed='left'), 1.0

In [362]:
data_age = pd.read_csv('../data/bayesian_network/age-fixed.csv')
data_health = pd.read_csv('../data/bayesian_network/health-fixed.csv')

data_age = data_age.rename(columns = {'gender': 'sex'})
data_age = data_age.set_index(['sex', 'age_group'])
data_age.index = data_age.index.set_levels(
    data_age.index.levels[data_age.index.names.index('age_group')].map(_str_to_interval),
    level='age_group'
)
data_age = data_age.sort_index()
data_age = data_age.rename(
    columns = {
        'Canada (except provinces)': 'Canada (excluding territories)'
    },
    index = {
        'Men+': 'Male',
        'Women+': 'Female'
    }
)

data_health = data_health.set_index(['sex', 'age_group'])
data_health.index = data_health.index.set_levels(
    data_health.index.levels[data_health.index.names.index('age_group')].map(_str_to_interval),
    level='age_group'
)
data_health = data_health.sort_index()

data_health = data_health.rename(
    columns = {
        'Body mass index, adjusted self-reported, adult (18 years and over), obese 15 16 17 18 19 20': 'Obese',
        'High blood pressure 21 22': 'High blood pressure',
        'Current smoker, daily or occasional 23 24 25 26 27': 'Current smoker',
        'Influenza immunization in the past 12 months 28 29': 'Recently vaccinated'
    },
    index = {
        'Males': 'Male',
        'Females': 'Female'
    }
)

In [363]:
_mapped = [_collapse_to_health_bracket(iv) for iv in data_age.index.get_level_values('age_group')]
_keep = [m is not None for m in _mapped]

_weights = np.array([m[1] for m in _mapped if m is not None])
_target_age = [m[0] for m in _mapped if m is not None]
_sex = data_age.index.get_level_values('sex')[_keep]

data_age = (
    data_age[_keep]
    .mul(_weights, axis=0)
    .set_axis(pd.MultiIndex.from_arrays([_sex, _target_age], names=['sex', 'age_group']))
    .groupby(level=['sex', 'age_group'])
    .sum()
    .round()
    .astype(int)
    .sort_index()
)

---------------------------

In [366]:
age_provinces = data_age.iloc[:, :-1]
age_canada = data_age['Canada (excluding territories)']

# P(location) - Probability of being from a province
p_loc = (age_provinces / age_canada.sum()).sum()

# P(sex|location)
p_sex_given_loc = age_provinces.groupby('sex').sum() / age_provinces.sum()

# P(age | sex, loc)
p_age_given_sexloc = age_provinces.groupby('sex').apply(lambda g: g / g.sum())

# P(condition | sex, age)
p_cond_given_sexage = data_health.div(age_canada, axis=0)

In [368]:
p_cond_given_sexage

Obese  High blood pressure  Current smoker  \
sex    age_group                                                     
Female [18.0, 35.0)  0.252204             0.021888        0.085431   
       [35.0, 50.0)  0.296601             0.080720        0.111373   
       [50.0, 65.0)  0.305924             0.235622        0.147570   
       [65.0, inf)   0.266461             0.433540        0.077563   
Male   [18.0, 35.0)  0.231669             0.029807        0.136498   
       [35.0, 50.0)  0.328482             0.115403        0.163680   
       [50.0, 65.0)  0.338450             0.297876        0.171227   
       [65.0, inf)   0.258559             0.428444        0.096822   

                     Recently vaccinated  
sex    age_group                          
Female [18.0, 35.0)             0.268530  
       [35.0, 50.0)             0.309622  
       [50.0, 65.0)             0.375195  
       [65.0, inf)              0.582319  
Male   [18.0, 35.0)             0.156944  
       [35.0, 50.0)             0.212139  
       [50.0, 65.0)             0.320271  
       [65.0, inf)              0.574659

In [381]:
def sample_single():
    loc = rng.choice(p_loc.index, p=p_loc.values)

    sexes = p_sex_given_loc[loc]
    sex = rng.choice(p_sex_given_loc.index, p=sexes.values)

    ages = p_age_given_sexloc[loc][sex]
    age = rng.choice(ages.index.get_level_values(1), p=ages.values)

    conds = p_cond_given_sexage.loc[(sex, age)]
    obese      = rng.random() < conds['Obese']
    smoker     = rng.random() < conds['Current smoker']
    hbp        = rng.random() < conds['High blood pressure']
    vaccinated = rng.random() < conds['Recently vaccinated']

    return (loc, sex, age, obese, smoker, hbp, vaccinated)

In [418]:
pd.DataFrame(
    [sample_single() for _ in range(10_000)],
    columns = ['location', 'sex', 'age', 'obese', 'smoker', 'hbp', 'vaccinated']
)

,location,sex,age,obese,smoker,hbp,vaccinated
0,Ontario,Male,"[50.0, 65.0)",False,False,False,False
1,Nova Scotia,Female,"[50.0, 65.0)",False,False,False,False
2,Saskatchewan,Female,"[65.0, inf)",True,False,False,True
3,Alberta,Female,"[18.0, 35.0)",False,False,False,False
4,Ontario,Male,"[50.0, 65.0)",True,False,False,False
...,...,...,...,...,...,...,...
9995,Quebec,Female,"[50.0, 65.0)",True,False,False,False
9996,Quebec,Male,"[50.0, 65.0)",True,False,True,False
9997,Ontario,Male,"[35.0, 50.0)",False,False,False,False
9998,Ontario,Male,"[50.0, 65.0)",False,False,False,False
